# 🏠 House Prices Prediction — TF-DF Gradient Boosted Trees

**Competition:** [House Prices: Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)  
**Model:** TensorFlow Decision Forests — `GradientBoostedTreesModel`  
**Metric:** RMSLE (Root Mean Squared Log Error)

> This notebook covers: outlier removal → feature engineering → log-target transform → GBT training → inverse-transform → submission.


## 0 · Environment Setup
Disable CUDA (TF-DF runs on CPU) and suppress verbose TF logs.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # Force CPU — TF-DF does not use GPU
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"    # Suppress TensorFlow C++ info/warning logs


## 1 · Imports

In [ ]:
import pandas as pd
import numpy as np
import tensorflow_decision_forests as tfdf


## 2 · Load Data

In [ ]:
TRAIN_PATH = "/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
TEST_PATH  = "/kaggle/input/house-prices-advanced-regression-techniques/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train shape : {train_df.shape}")
print(f"Test  shape : {test_df.shape}")


## 3 · Outlier Removal

Two well-known outliers in the Ames dataset have very large living area (> 4 000 sq ft)
but unusually low sale prices. Removing them improves generalisation.


In [ ]:
outlier_mask = (train_df["GrLivArea"] > 4000) & (train_df["SalePrice"] < 300_000)
print(f"Dropping {outlier_mask.sum()} outlier row(s).")
train_df = train_df.drop(train_df[outlier_mask].index).reset_index(drop=True)


## 4 · Feature Engineering

Three hand-crafted features that capture the most important price signals:

| Feature | Formula | Rationale |
|---------|---------|-----------|
| `TotalSF` | `GrLivArea + TotalBsmtSF` | Overall living space |
| `TotalBath` | full baths + 0.5 × half baths (above + below grade) | Bathroom count proxy |
| `Age` | `YrSold − YearBuilt` | House age at sale time |

Column names are also sanitised (`.` and spaces → `_`) to avoid TF-DF parsing errors.


In [ ]:
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add domain-driven features and sanitise column names."""
    df = df.copy()

    # Total living area (above grade + basement)
    df["TotalSF"] = df["GrLivArea"] + df["TotalBsmtSF"]

    # Weighted bathroom count (half-baths count as 0.5)
    df["TotalBath"] = (
        df["FullBath"]
        + 0.5 * df["HalfBath"]
        + df["BsmtFullBath"]
        + 0.5 * df["BsmtHalfBath"]
    )

    # Age of the house at the time of sale
    df["Age"] = df["YrSold"] - df["YearBuilt"]

    # Sanitise column names — TF-DF rejects dots and spaces
    df.columns = [c.replace(".", "_").replace(" ", "_") for c in df.columns]

    return df

train_df = create_features(train_df)
test_df  = create_features(test_df)
print("New features added:", ["TotalSF", "TotalBath", "Age"])


## 5 · Log-Transform Target & Build TF Datasets

The competition metric is **RMSLE**, so we train on `log1p(SalePrice)` and
invert with `expm1` at prediction time.

`Id` is saved separately and dropped before building the TF datasets — leaving
it in would cause a feature-column mismatch between train and test.


In [ ]:
# Log-transform the target (matches the RMSLE competition metric)
train_df["SalePrice"] = np.log1p(train_df["SalePrice"])

# Preserve test IDs for the submission file
test_ids = test_df["Id"].values

# Build TF datasets — drop Id from both splits
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    train_df.drop("Id", axis=1),
    label="SalePrice",
    task=tfdf.keras.Task.REGRESSION,
)

test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    test_df.drop("Id", axis=1),
    task=tfdf.keras.Task.REGRESSION,
)


## 6 · Model Definition & Training

Key hyperparameters:

| Parameter | Value | Notes |
|-----------|-------|-------|
| `num_trees` | 2 000 | More trees → better ensemble |
| `shrinkage` | 0.02 | Low learning rate reduces overfitting |
| `max_depth` | 6 | Controls individual tree complexity |
| `growing_strategy` | `BEST_FIRST_GLOBAL` | Grows leaves greedily across the whole tree |
| `l1_regularization` | 0.05 | L1 penalty — encourages sparsity |
| `l2_regularization` | 0.05 | L2 penalty — shrinks large weights |


In [ ]:
model = tfdf.keras.GradientBoostedTreesModel(
    task=tfdf.keras.Task.REGRESSION,
    num_trees=2000,
    shrinkage=0.02,           # Learning rate
    max_depth=6,
    growing_strategy="BEST_FIRST_GLOBAL",
    min_examples=5,
    l1_regularization=0.05,
    l2_regularization=0.05,
)

model.fit(train_ds)
print("Training complete.")


## 7 · Predict & Inverse-Transform

Predictions are in log-space → apply `expm1` to recover real prices.
Any negative values (numerically impossible) are clipped to 0.


In [ ]:
preds = model.predict(test_ds)
final_preds = np.expm1(preds.flatten())          # Reverse log1p
final_preds = np.clip(final_preds, a_min=0, a_max=None)  # Guard against negatives

print(f"Prediction stats — min: {final_preds.min():,.0f}  "
      f"mean: {final_preds.mean():,.0f}  max: {final_preds.max():,.0f}")


## 8 · Create Submission File

In [ ]:
submission = pd.DataFrame({"Id": test_ids, "SalePrice": final_preds})
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("submission.csv saved — ready to submit!")
submission.head()
